# Medical Abstract Sentence Classification

This notebook builds a medical abstract sentence classifier for the PubMed 20k RCT dataset. The goal is to predict whether each sentence is `BACKGROUND`, `OBJECTIVE`, `METHODS`, `RESULTS`, or `CONCLUSIONS`.

The main experiment compares text-only TF-IDF baselines with a model that adds abstract structure features: `line_number`, `total_lines`, and `relative_position`.


In [ ]:
# Core data libraries
import pandas as pd
import numpy as np

# Plotting and notebook display utilities
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

# Path handling and saved model artifacts
from pathlib import Path
import json
import joblib

# Scikit-learn utilities for features, models, and evaluation
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC


In [ ]:
# Resolve the project root whether this notebook is run from the root folder or notebooks/.
CURRENT_DIR = Path.cwd().resolve()
ROOT_DIR = CURRENT_DIR if (CURRENT_DIR / "src").exists() else CURRENT_DIR.parent

DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw" / "20k_abstracts"
OUTPUT_DIR = ROOT_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
MODEL_DIR = OUTPUT_DIR / "models"
PREDICTION_DIR = OUTPUT_DIR / "predictions"

# Create output folders so every table, plot, model, and prediction is saved separately.
for directory in [FIGURE_DIR, TABLE_DIR, MODEL_DIR, PREDICTION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
def get_lines(filepath):
    """Read a PubMed RCT text file and return its raw lines."""
    with open(filepath, "r", encoding="utf-8") as file:
        return file.readlines()

# Load raw text files from the 20k split.
train_lines = get_lines(RAW_DATA_DIR / "train.txt")
val_lines = get_lines(RAW_DATA_DIR / "dev.txt")
test_lines = get_lines(RAW_DATA_DIR / "test.txt")


In [ ]:
# Save a compact summary of raw line counts for each split.
raw_line_counts = pd.DataFrame(
    [
        {"split": "train", "raw_line_count": len(train_lines)},
        {"split": "validation", "raw_line_count": len(val_lines)},
        {"split": "test", "raw_line_count": len(test_lines)},
    ]
)
raw_line_counts.to_csv(TABLE_DIR / "raw_line_counts.csv", index=False)


In [ ]:
# Load the saved table in a separate display cell.
pd.read_csv(TABLE_DIR / "raw_line_counts.csv")


In [ ]:
def preprocess_text_with_line_numbers(filepath):
    """Parse a PubMed RCT split into sentence-level dictionaries.

    The file format uses abstract ID lines beginning with ###, labeled sentence
    lines separated by tabs, and blank lines between abstracts.
    """
    input_lines = get_lines(filepath)
    abstract_lines = []
    abstract_samples = []

    def flush_abstract(lines):
        total_lines = len(lines)
        if total_lines == 0:
            return
        for line_number, abstract_line in enumerate(lines):
            if "\t" not in abstract_line:
                continue
            target, text = abstract_line.split("\t", maxsplit=1)
            abstract_samples.append(
                {
                    "target": target.strip(),
                    "text": text.strip(),
                    "line_number": line_number,
                    "total_lines": total_lines,
                    "relative_position": line_number / total_lines,
                }
            )

    for line in input_lines:
        stripped_line = line.strip()
        if stripped_line.startswith("###"):
            flush_abstract(abstract_lines)
            abstract_lines = []
        elif stripped_line == "":
            flush_abstract(abstract_lines)
            abstract_lines = []
        else:
            abstract_lines.append(stripped_line)

    flush_abstract(abstract_lines)
    return abstract_samples


In [ ]:
# Convert the parsed examples into DataFrames with standard columns.
train_df = pd.DataFrame(preprocess_text_with_line_numbers(RAW_DATA_DIR / "train.txt"))
val_df = pd.DataFrame(preprocess_text_with_line_numbers(RAW_DATA_DIR / "dev.txt"))
test_df = pd.DataFrame(preprocess_text_with_line_numbers(RAW_DATA_DIR / "test.txt"))

standard_columns = ["target", "text", "line_number", "total_lines", "relative_position"]
train_df = train_df[standard_columns]
val_df = val_df[standard_columns]
test_df = test_df[standard_columns]

# Save readable previews rather than relying only on notebook output.
train_df.head(20).to_csv(TABLE_DIR / "train_processed_preview.csv", index=False)
val_df.head(20).to_csv(TABLE_DIR / "val_processed_preview.csv", index=False)
test_df.head(20).to_csv(TABLE_DIR / "test_processed_preview.csv", index=False)


In [ ]:
# Display the saved processed training preview.
pd.read_csv(TABLE_DIR / "train_processed_preview.csv").head()


In [ ]:
# Calculate label distribution for train, validation, and test splits.
label_order = ["BACKGROUND", "OBJECTIVE", "METHODS", "RESULTS", "CONCLUSIONS"]
splits = {"train": train_df, "validation": val_df, "test": test_df}

records = []
for split_name, df in splits.items():
    counts = df["target"].value_counts().reindex(label_order, fill_value=0)
    for label, count in counts.items():
        records.append(
            {
                "split": split_name,
                "target": label,
                "count": int(count),
                "percentage": count / len(df),
            }
        )

label_distribution = pd.DataFrame(records)
label_distribution.to_csv(TABLE_DIR / "label_distribution.csv", index=False)

plot_df = label_distribution.pivot(index="target", columns="split", values="count").loc[label_order]
ax = plot_df.plot(kind="bar", figsize=(9, 5), color=["#3366CC", "#DC3912", "#109618"])
ax.set_title("Label Distribution by Split")
ax.set_xlabel("Sentence role")
ax.set_ylabel("Number of sentences")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "label_distribution.png", dpi=160, bbox_inches="tight")
plt.close()


In [ ]:
# Display the saved label distribution table and figure.
display(pd.read_csv(TABLE_DIR / "label_distribution.csv"))
display(Image(filename=str(FIGURE_DIR / "label_distribution.png")))


In [ ]:
# Add sentence length as a simple EDA feature and summarize by split.
length_records = []
plt.figure(figsize=(9, 5))

for split_name, df in splits.items():
    df["sentence_length"] = df["text"].str.split().map(len)
    lengths = df["sentence_length"]
    length_records.append(
        {
            "split": split_name,
            "count": int(lengths.count()),
            "mean_words": lengths.mean(),
            "median_words": lengths.median(),
            "min_words": lengths.min(),
            "max_words": lengths.max(),
            "p90_words": lengths.quantile(0.90),
        }
    )
    plt.hist(lengths, bins=50, alpha=0.45, label=split_name)

text_length_summary = pd.DataFrame(length_records)
text_length_summary.to_csv(TABLE_DIR / "text_length_summary.csv", index=False)

plt.title("Sentence Length Distribution")
plt.xlabel("Words per sentence")
plt.ylabel("Number of sentences")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "text_length_histogram.png", dpi=160, bbox_inches="tight")
plt.close()


In [ ]:
# Display the saved sentence length table and figure.
display(pd.read_csv(TABLE_DIR / "text_length_summary.csv"))
display(Image(filename=str(FIGURE_DIR / "text_length_histogram.png")))


In [ ]:
# Prepare text inputs and labels for the baseline models.
X_train_text = train_df["text"]
X_val_text = val_df["text"]
X_test_text = test_df["text"]

y_train = train_df["target"]
y_val = val_df["target"]
y_test = test_df["target"]

# Fit a label encoder for a saved mapping table, even though sklearn models can use string labels directly.
label_encoder = LabelEncoder()
label_encoder.fit(y_train)
label_mapping = pd.DataFrame(
    {
        "encoded_label": range(len(label_encoder.classes_)),
        "target": label_encoder.classes_,
    }
)
label_mapping.to_csv(TABLE_DIR / "label_mapping.csv", index=False)


In [ ]:
# Display the saved label mapping.
pd.read_csv(TABLE_DIR / "label_mapping.csv")


In [ ]:
def calculate_metrics(y_true, y_pred):
    """Calculate the main evaluation metrics for this project."""
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

# Baseline 1: TF-IDF text features with Logistic Regression.
logreg_model = Pipeline(
    steps=[
        ("tfidf", TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("classifier", OneVsRestClassifier(LogisticRegression(solver="liblinear", max_iter=1000, class_weight="balanced", random_state=42))),
    ]
)

logreg_model.fit(X_train_text, y_train)
logreg_val_predictions = logreg_model.predict(X_val_text)
logreg_metrics = calculate_metrics(y_val, logreg_val_predictions)

joblib.dump(logreg_model, MODEL_DIR / "tfidf_logreg.joblib")
pd.DataFrame([{ "model": "TF-IDF + Logistic Regression", **logreg_metrics }]).to_csv(
    TABLE_DIR / "logreg_validation_metrics.csv", index=False
)
pd.DataFrame(
    classification_report(y_val, logreg_val_predictions, labels=label_order, output_dict=True, zero_division=0)
).transpose().reset_index().rename(columns={"index": "label"}).to_csv(
    TABLE_DIR / "logreg_classification_report.csv", index=False
)


In [ ]:
# Display saved Logistic Regression results.
display(pd.read_csv(TABLE_DIR / "logreg_validation_metrics.csv"))
display(pd.read_csv(TABLE_DIR / "logreg_classification_report.csv"))


In [ ]:
# Baseline 2: TF-IDF text features with LinearSVC.
linearsvc_model = Pipeline(
    steps=[
        ("tfidf", TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("classifier", LinearSVC(class_weight="balanced", dual="auto", random_state=42, max_iter=5000)),
    ]
)

linearsvc_model.fit(X_train_text, y_train)
linearsvc_val_predictions = linearsvc_model.predict(X_val_text)
linearsvc_metrics = calculate_metrics(y_val, linearsvc_val_predictions)

joblib.dump(linearsvc_model, MODEL_DIR / "tfidf_linearsvc.joblib")
pd.DataFrame([{ "model": "TF-IDF + LinearSVC", **linearsvc_metrics }]).to_csv(
    TABLE_DIR / "linearsvc_validation_metrics.csv", index=False
)
pd.DataFrame(
    classification_report(y_val, linearsvc_val_predictions, labels=label_order, output_dict=True, zero_division=0)
).transpose().reset_index().rename(columns={"index": "label"}).to_csv(
    TABLE_DIR / "linearsvc_classification_report.csv", index=False
)


In [ ]:
# Display saved LinearSVC results.
display(pd.read_csv(TABLE_DIR / "linearsvc_validation_metrics.csv"))
display(pd.read_csv(TABLE_DIR / "linearsvc_classification_report.csv"))


In [ ]:
# Structural features are useful because medical abstracts usually follow a logical order:
# Background -> Objective -> Methods -> Results -> Conclusions.
# This transformer combines sentence text with sentence position information.
position_features = ["line_number", "total_lines", "relative_position"]

text_and_position_transformer = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2, sublinear_tf=True), "text"),
        ("position", StandardScaler(), position_features),
    ]
)

X_train_position = train_df[["text", *position_features]]
X_val_position = val_df[["text", *position_features]]
X_test_position = test_df[["text", *position_features]]


In [ ]:
# Model 3: TF-IDF plus structural position features with Logistic Regression.
position_logreg_model = Pipeline(
    steps=[
        ("features", text_and_position_transformer),
        ("classifier", OneVsRestClassifier(LogisticRegression(solver="liblinear", max_iter=1000, class_weight="balanced", random_state=42))),
    ]
)

position_logreg_model.fit(X_train_position, y_train)
position_val_predictions = position_logreg_model.predict(X_val_position)
position_logreg_metrics = calculate_metrics(y_val, position_val_predictions)

joblib.dump(position_logreg_model, MODEL_DIR / "tfidf_position_logreg.joblib")
pd.DataFrame([{ "model": "TF-IDF + Position Logistic Regression", **position_logreg_metrics }]).to_csv(
    TABLE_DIR / "position_logreg_validation_metrics.csv", index=False
)
pd.DataFrame(
    classification_report(y_val, position_val_predictions, labels=label_order, output_dict=True, zero_division=0)
).transpose().reset_index().rename(columns={"index": "label"}).to_csv(
    TABLE_DIR / "position_logreg_classification_report.csv", index=False
)


In [ ]:
# Display saved structural feature model results.
display(pd.read_csv(TABLE_DIR / "position_logreg_validation_metrics.csv"))
display(pd.read_csv(TABLE_DIR / "position_logreg_classification_report.csv"))


In [ ]:
# Combine validation metrics from all models into one comparison table.
model_comparison = pd.concat(
    [
        pd.read_csv(TABLE_DIR / "logreg_validation_metrics.csv"),
        pd.read_csv(TABLE_DIR / "linearsvc_validation_metrics.csv"),
        pd.read_csv(TABLE_DIR / "position_logreg_validation_metrics.csv"),
    ],
    ignore_index=True,
).sort_values("macro_f1", ascending=False)

model_comparison.to_csv(TABLE_DIR / "model_comparison.csv", index=False)

ax = model_comparison.set_index("model")[["accuracy", "macro_f1", "weighted_f1"]].plot(
    kind="bar", figsize=(10, 5), color=["#3366CC", "#DC3912", "#109618"]
)
ax.set_title("Validation Model Comparison")
ax.set_xlabel("Model")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "model_comparison.png", dpi=160, bbox_inches="tight")
plt.close()


In [ ]:
# Display saved model comparison outputs.
display(pd.read_csv(TABLE_DIR / "model_comparison.csv"))
display(Image(filename=str(FIGURE_DIR / "model_comparison.png")))


In [ ]:
# Select the best validation model based on macro F1 and evaluate it on the test set.
best_model_name = pd.read_csv(TABLE_DIR / "model_comparison.csv").iloc[0]["model"]

model_registry = {
    "TF-IDF + Logistic Regression": (logreg_model, "text"),
    "TF-IDF + LinearSVC": (linearsvc_model, "text"),
    "TF-IDF + Position Logistic Regression": (position_logreg_model, "position"),
}
best_model, best_input_mode = model_registry[best_model_name]

X_test_best = X_test_position if best_input_mode == "position" else X_test_text
test_predictions = best_model.predict(X_test_best)
test_metrics = calculate_metrics(y_test, test_predictions)

joblib.dump(best_model, MODEL_DIR / "best_model.joblib")
metadata = {
    "best_model_name": best_model_name,
    "model_filename": "best_model.joblib",
    "input_mode": best_input_mode,
    "labels": label_order,
}
(MODEL_DIR / "best_model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

pd.DataFrame([{ "model": best_model_name, **test_metrics }]).to_csv(
    TABLE_DIR / "best_model_test_metrics.csv", index=False
)
pd.DataFrame(
    classification_report(y_test, test_predictions, labels=label_order, output_dict=True, zero_division=0)
).transpose().reset_index().rename(columns={"index": "label"}).to_csv(
    TABLE_DIR / "best_model_test_classification_report.csv", index=False
)


In [ ]:
# Display saved test set results.
display(pd.read_csv(TABLE_DIR / "best_model_test_metrics.csv"))
display(pd.read_csv(TABLE_DIR / "best_model_test_classification_report.csv"))


In [ ]:
# Generate and save a confusion matrix for the best test model.
cm = confusion_matrix(y_test, test_predictions, labels=label_order)
confusion_matrix_df = pd.DataFrame(cm, index=label_order, columns=label_order)
confusion_matrix_df.index.name = "true_label"
confusion_matrix_df.to_csv(TABLE_DIR / "confusion_matrix.csv")

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_order).plot(
    ax=ax, cmap="Blues", values_format="d", colorbar=False
)
ax.set_title("Best Model Confusion Matrix")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.close()


In [ ]:
# Display the saved confusion matrix table and figure.
display(pd.read_csv(TABLE_DIR / "confusion_matrix.csv"))
display(Image(filename=str(FIGURE_DIR / "confusion_matrix.png")))


In [ ]:
# Save the first 30 misclassified examples for readable error analysis.
misclassified_mask = y_test.to_numpy() != test_predictions
misclassified_examples = test_df.loc[
    misclassified_mask,
    ["text", "line_number", "total_lines", "relative_position"],
].copy()
misclassified_examples.insert(1, "true_label", y_test.to_numpy()[misclassified_mask])
misclassified_examples.insert(2, "predicted_label", test_predictions[misclassified_mask])

misclassified_examples.head(30).to_csv(PREDICTION_DIR / "misclassified_examples.csv", index=False)


In [ ]:
# Display saved error examples and a short interpretation note.
display(pd.read_csv(PREDICTION_DIR / "misclassified_examples.csv"))
display(
    Markdown(
        """
**Error analysis notes**

- Background vs Objective may be confused because both appear near the beginning of abstracts.
- Methods vs Results may be confused because both contain experimental terms.
- Position features can help, but cannot fully solve semantic overlap.
"""
    )
)


In [ ]:
# Interpret the text-only Logistic Regression model by extracting top positive TF-IDF words per class.
vectorizer = logreg_model.named_steps["tfidf"]
classifier = logreg_model.named_steps["classifier"]
feature_names = np.asarray(vectorizer.get_feature_names_out())

top_word_records = []
for class_index, class_label in enumerate(classifier.classes_):
    if hasattr(classifier, "coef_"):
        coefficients = classifier.coef_[class_index]
    else:
        coefficients = classifier.estimators_[class_index].coef_.ravel()
    top_indices = np.argsort(coefficients)[-15:][::-1]
    for rank, feature_index in enumerate(top_indices, start=1):
        top_word_records.append(
            {
                "target": class_label,
                "rank": rank,
                "word_or_phrase": feature_names[feature_index],
                "coefficient": coefficients[feature_index],
            }
        )

top_words_by_class = pd.DataFrame(top_word_records)
top_words_by_class.to_csv(TABLE_DIR / "top_words_by_class.csv", index=False)


In [ ]:
# Display saved top words by class.
pd.read_csv(TABLE_DIR / "top_words_by_class.csv")


In [ ]:
def predict_sentence_role(sentence, line_number=None, total_lines=None):
    """Load the saved best model and predict one sentence role."""
    metadata = json.loads((MODEL_DIR / "best_model_metadata.json").read_text(encoding="utf-8"))
    model = joblib.load(MODEL_DIR / metadata.get("model_filename", "best_model.joblib"))
    input_mode = metadata.get("input_mode", "text")

    if input_mode == "position":
        safe_line_number = 0 if line_number is None else line_number
        safe_total_lines = 1 if total_lines in (None, 0) else total_lines
        model_input = pd.DataFrame(
            [
                {
                    "text": sentence,
                    "line_number": safe_line_number,
                    "total_lines": safe_total_lines,
                    "relative_position": safe_line_number / safe_total_lines,
                }
            ]
        )[["text", "line_number", "total_lines", "relative_position"]]
    else:
        model_input = [sentence]

    return model.predict(model_input)[0]

# Save several demo predictions for easy review.
demo_sentences = pd.DataFrame(
    [
        {"text": "The aim of this study was to evaluate the effect of treatment.", "line_number": 1, "total_lines": 10},
        {"text": "Participants were randomly assigned to receive placebo or active medication.", "line_number": 4, "total_lines": 10},
        {"text": "The intervention group showed a statistically significant reduction in symptoms.", "line_number": 7, "total_lines": 10},
        {"text": "These findings suggest that the treatment may improve patient outcomes.", "line_number": 9, "total_lines": 10},
    ]
)
demo_sentences["predicted_label"] = demo_sentences.apply(
    lambda row: predict_sentence_role(row["text"], row["line_number"], row["total_lines"]), axis=1
)
demo_sentences.to_csv(PREDICTION_DIR / "demo_predictions.csv", index=False)


In [ ]:
# Display saved demo predictions.
pd.read_csv(PREDICTION_DIR / "demo_predictions.csv")


## Conclusion

This project built a complete sentence classification workflow for PubMed 20k RCT medical abstracts. It includes data parsing, EDA, TF-IDF baselines, a structural feature model, validation comparison, test evaluation, confusion matrix analysis, error analysis, feature interpretation, and a prediction function.

The best model is selected by validation macro F1. This task is harder than simple sentiment classification because labels depend on scientific discourse structure, not just positive or negative wording. Sentences can share similar medical vocabulary while playing different roles in the abstract.

Remaining limitations include independent sentence classification, limited semantic understanding from TF-IDF, and evaluation only on PubMed 20k RCT. Future work should include SciBERT or BioBERT fine-tuning, abstract-level sequence modeling, and experiments on the full PubMed 200k RCT dataset.
